In [4]:
# Importações básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Importações do Scikit-Learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Importação do csv
df_batalhas = pd.read_csv('dados/batalhas_pokemon_treino.csv')
print(f"Dataset de batalhas carregado: {df_batalhas.shape[0]} linhas e {df_batalhas.shape[1]} colunas.")

ModuleNotFoundError: No module named 'pandas'

Divisão em Atributos (Features) e Alvo (Target) + Treino/Teste

Estamos aqui dizendo ao modelo o que ele vai usar para prever (X: os status dos dois Pokémons) e o que ele deve adivinhar (y: quem ganhou). Também separamos 80% dos dados para o modelo estudar (Treino) e 20% para testar se ele realmente aprendeu (Teste).

In [19]:
# Separando as variáveis preditoras (X) e a variável alvo (y)
X = df_batalhas.drop(columns=['Vitoria_P1'])
y = df_batalhas['Vitoria_P1']

# Divisão em Treino (80%) e Teste (20%) - Corrigido para 'test_size'
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dados de Treino: {X_train.shape[0]} amostras")
print(f"Dados de Teste: {X_test.shape[0]} amostras")

Dados de Treino: 8000 amostras
Dados de Teste: 2000 amostras


Método Supervisionado 1 - Regressão Logística

A Regressão Logística busca encontrar uma linha de decisão matemática linear que separa as vitórias das derrotas.

In [20]:
print("--- Treinando Regressão Logística ---")
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

# Predição
y_pred_lr = model_lr.predict(X_test)

--- Treinando Regressão Logística ---


Método Supervisionado 2 - Random Forest + Ajuste de Parâmetros

O Random Forest cria várias "Árvores de Decisão" e combina os votos delas. Já incluímos aqui o GridSearchCV.

In [ ]:
print("--- Treinando Random Forest com GridSearchCV ---")

# Definição dos parâmetros (PARA NA HORA DE ESCREVER O RELATÓRIO SABER QUE AQUI É O 'Ajuste de hiperparâmetros (quando aplicável)')
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

rf_base = RandomForestClassifier(random_state=42)

# O GridSearchCV vai testar as combinações cruzando os dados para achar a melhor
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Melhor modelo encontrado
model_rf = grid_search.best_estimator_
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")

# Predição
y_pred_rf = model_rf.predict(X_test)

--- Treinando Random Forest com GridSearchCV ---


Melhores parâmetros encontrados: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}


Método Supervisionado 3 - Rede Neural (Deep Learning)

A Rede Neural tenta simular sinapses para encontrar relações ocultas entre os status, como perceber por exemplo que Speed Alto + Attack Alto é algo fatal.

In [28]:
print("--- Treinando Rede Neural (MLPClassifier) ---")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_nn = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
model_nn.fit(X_train_scaled, y_train)

# 3. Predição
y_pred_nn = model_nn.predict(X_test_scaled)


--- Treinando Rede Neural (MLPClassifier) ---


Aplicação Prática - Recomendador de XP.

Criamos uma função que dado um Pokémon, o script faz o modelo de ML prever a chance dele ganhar contra TODOS os outros Pokémons (1.215), ele filtra quais ele tem mais de 85% de chance de vencer (ou seja, uma vitória segura) e ordena pelo maior ganho de experiência.

In [ ]:
from dados_processados import limpar_dados_pokemon 
from utils import get_effectiveness

def recomendar_oponente_para_farmar_xp(nome_meu_pokemon, top_n=5):
    # Carregar o dataset original limpo dos Pokémons
    df_pk = limpar_dados_pokemon()
    
    # Verificar se o Pokémon existe na base
    nome_meu_pokemon = nome_meu_pokemon.title()
    if nome_meu_pokemon not in df_pk['Pokemon'].values:
        print(f"Pokémon '{nome_meu_pokemon}' não encontrado na base de dados.")
        return
        
    meu_pk = df_pk[df_pk['Pokemon'] == nome_meu_pokemon].iloc[0]
    
    lista_predicoes = []
    
    # Comparar contra todos os outros Pokémons do jogo
    for _, oponente in df_pk.iterrows():
        if oponente['Pokemon'] == meu_pk['Pokemon']:
            continue
            
        # Calcula a vantagem que o meu Pokémon tem contra o oponente
        eff_meu = max(get_effectiveness(meu_pk['Type1'], oponente['Type1'], oponente['Type2']),
                      get_effectiveness(meu_pk['Type2'], oponente['Type1'], oponente['Type2']))
                      
        # Calcula a vantagem que o oponente tem contra o meu Pokémon
        eff_opo = max(get_effectiveness(oponente['Type1'], meu_pk['Type1'], meu_pk['Type2']),
                      get_effectiveness(oponente['Type2'], meu_pk['Type1'], meu_pk['Type2']))
            
        # Montar a linha de features incluindo as colunas numéricas de vantagem
        features_combate = pd.DataFrame([{
            'P1_HP': meu_pk['HP Max'], 'P1_Atk': meu_pk['Attack Max'], 'P1_Def': meu_pk['Defense Max'], 
            'P1_SpAtk': meu_pk['Special Attack Max'], 'P1_SpDef': meu_pk['Special Defense Max'], 'P1_Speed': meu_pk['Speed Max'],
            'P2_HP': oponente['HP Max'], 'P2_Atk': oponente['Attack Max'], 'P2_Def': oponente['Defense Max'], 
            'P2_SpAtk': oponente['Special Attack Max'], 'P2_SpDef': oponente['Special Defense Max'], 'P2_Speed': oponente['Speed Max'],
            'Vantagem_Tipo_P1': eff_meu,
            'Vantagem_Tipo_P2': eff_opo
        }])
        
        # GARANTIA: Forçar a mesma ordem de colunas usada no treinamento
        features_combate = features_combate[X.columns]
        
        # --- PREVISÕES DOS MODELOS (predict_proba pega a chance de vitória) ---
        prob_rf = model_rf.predict_proba(features_combate)[0][1] # Random Forest
        prob_lr = model_lr.predict_proba(features_combate)[0][1] # Regressão Logística
        
        # Para a Rede Neural, escalamos os dados antes.
        features_scaled = scaler.transform(features_combate)
        prob_nn = model_nn.predict_proba(features_scaled)[0][1]  # Rede Neural
        
        lista_predicoes.append({
            'Oponente': oponente['Pokemon'],
            'Tipos_Oponente': oponente['Type'],
            'Chance_RF': prob_rf,
            'Chance_LR': prob_lr,
            'Chance_NN': prob_nn,
            'XP_Recompensa': oponente['Base Exp']
        })
        
    df_resultados = pd.DataFrame(lista_predicoes)
    
    # Filtrar apenas lutas com mais de 85% de chance de vitória no modelo principal (Random Forest)
    df_seguro = df_resultados[df_resultados['Chance_RF'] >= 0.85]
    
    # Ordenar pelo maior ganho de XP
    df_ranking = df_seguro.sort_values(by='XP_Recompensa', ascending=False).head(top_n)
    
    print(f"=== RECOMENDAÇÕES DE FARMING PARA: {nome_meu_pokemon.upper()} ===")
    if df_ranking.empty:
        print("Infelizmente não encontramos oponentes seguros que garantam vitória fácil para este Pokémon.")
    else:
        for idx, row in df_ranking.iterrows():
            print(f"\nOponente: {row['Oponente']} ({row['Tipos_Oponente']}) | XP Obtido: {row['XP_Recompensa']}")
            print(f"  -> Chance RF (Árvore de Decisão): {row['Chance_RF']*100:.1f}%")
            print(f"  -> Chance LR (Regress. Logística): {row['Chance_LR']*100:.1f}%")
            print(f"  -> Chance NN (Rede Neural):        {row['Chance_NN']*100:.1f}%")

# --- TESTANDO O RECOMENDADOR ---
recomendar_oponente_para_farmar_xp('Gengar')

=== RECOMENDAÇÕES DE FARMING PARA: SUNKERN ===

Oponente: Finneon (Water) | XP Obtido: 66
  -> Chance RF (Árvore de Decisão): 86.4%
  -> Chance LR (Regress. Logística): 99.6%
  -> Chance NN (Rede Neural):        100.0%

Oponente: Froakie (Water) | XP Obtido: 63
  -> Chance RF (Árvore de Decisão): 85.8%
  -> Chance LR (Regress. Logística): 99.8%
  -> Chance NN (Rede Neural):        100.0%

Oponente: Squirtle (Water) | XP Obtido: 63
  -> Chance RF (Árvore de Decisão): 85.4%
  -> Chance LR (Regress. Logística): 99.6%
  -> Chance NN (Rede Neural):        100.0%


Temos a preferência ao Random Forest na função final porque, em problemas complexos com muitas variáveis (como status de RPG), algoritmos baseados em árvores de decisão tendem a ter um desempenho absurdamente superior a algoritmos lineares, mas estamos mesmo assim colocando a resposta de todos os modelos.

O de baixo estou testando dois pokémons específicos, onde é MUITO difícil o Gengar vencer o MewTwo, estou fazendo isso para ver se o pokémon1 (P1), não está com alguma vantagem por algum erro cometido por mim.

In [33]:
from dados_processados import limpar_dados_pokemon
from utils import get_effectiveness
import pandas as pd

# Carregando a base explicitamente para esta célula
df_pk = limpar_dados_pokemon()

meu = df_pk[df_pk['Pokemon'] == 'Gengar'].iloc[0]
opo = df_pk[df_pk['Pokemon'] == 'Mewtwo'].iloc[0]

eff_meu = max(get_effectiveness(meu['Type1'], opo['Type1'], opo['Type2']), get_effectiveness(meu['Type2'], opo['Type1'], opo['Type2']))
eff_opo = max(get_effectiveness(opo['Type1'], meu['Type1'], meu['Type2']), get_effectiveness(opo['Type2'], meu['Type1'], meu['Type2']))

teste_combate = pd.DataFrame([{
    'P1_HP': meu['HP Max'], 'P1_Atk': meu['Attack Max'], 'P1_Def': meu['Defense Max'], 
    'P1_SpAtk': meu['Special Attack Max'], 'P1_SpDef': meu['Special Defense Max'], 'P1_Speed': meu['Speed Max'],
    'P2_HP': opo['HP Max'], 'P2_Atk': opo['Attack Max'], 'P2_Def': opo['Defense Max'], 
    'P2_SpAtk': opo['Special Attack Max'], 'P2_SpDef': opo['Special Defense Max'], 'P2_Speed': opo['Speed Max'],
    'Vantagem_Tipo_P1': eff_meu, 'Vantagem_Tipo_P2': eff_opo
}])[X.columns]

print(f"Chance do Gengar vencer o Mewtwo:")
print(f"Random Forest: {model_rf.predict_proba(teste_combate)[0][1]*100:.1f}%")
print(f"Regressão Logística: {model_lr.predict_proba(teste_combate)[0][1]*100:.1f}%")
print(f"Rede Neural: {model_nn.predict_proba(scaler.transform(teste_combate))[0][1]*100:.1f}%")

Chance do Gengar vencer o Mewtwo:
Random Forest: 6.0%
Regressão Logística: 0.1%
Rede Neural: 0.0%
